<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px; float: right;">
    </div>
</a>

# **Factory Optimization with cuOpt**

<h2><b>Review Artifact:</b> Possible Objective Swaps</h2>

This notebook is a decision menu for UME review.

It is not yet part of the required student path. We will decide later whether each objective belongs in the autoclave MIP notebook, the drill-routing notebook, the formulation-extension notebooks, or the PPT discussion.

## Goal

An objective swap means:

> Keep most of the same data and constraints, but change what the solver is trying to optimize.

This is one of the best teaching moves in optimization because it shows students that **feasible** and **best** are different ideas.

Use this notebook to decide which objective swaps are worth adding to the workshop.

## Setup

The tables below are lightweight review tables. They do not solve a model.

The actual model code would be moved into the right notebook after UME picks which objectives matter most.

In [ ]:
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 120)

<hr>

## Autoclave MIP Objective Swaps

For the autoclave model, the decisions are binary assignment decisions:

> `x[p, a, r, t] = 1` if part `p` is cured on autoclave `a`, using recipe `r`, starting at time `t`.

Changing the objective changes which feasible schedule cuOpt prefers.

In [ ]:
mip_objectives = pd.DataFrame([
    {
        "objective": "Maximize throughput",
        "plain_english": "Cure as many parts as possible before the out-time deadline.",
        "formulation_shape": "MAX sum(x[p, a, r, t])",
        "new_data_needed": "none",
        "best_location": "Already in notebook 01 as the base idea",
    },
    {
        "objective": "Maximize priority-weighted throughput",
        "plain_english": "If capacity is tight, prefer higher-priority parts.",
        "formulation_shape": "MAX sum((1000 + priority[p]) * x[p, a, r, t])",
        "new_data_needed": "priority already exists",
        "best_location": "Notebook 01 or 02",
    },
    {
        "objective": "Minimize lateness",
        "plain_english": "Schedule every part, then make late parts as little late as possible.",
        "formulation_shape": "MIN sum(weight[p] * lateness[p])",
        "new_data_needed": "none if deadline_hr is used as the target",
        "best_location": "Notebook 02 as the soft-deadline extension",
    },
    {
        "objective": "Minimize scrap or waste cost",
        "plain_english": "Missing a high-value part should hurt more than missing a low-value part.",
        "formulation_shape": "MIN sum(scrap_cost[p] * missed[p])",
        "new_data_needed": "scrap_cost, part_value, or recovery_cost",
        "best_location": "PPT discussion first; notebook later if UME has data",
    },
    {
        "objective": "Minimize extra autoclave use",
        "plain_english": "Use the second autoclave only when the throughput gain is worth it.",
        "formulation_shape": "MIN sum(extra_autoclave_used[a]) or MAX throughput - penalty * extra_use",
        "new_data_needed": "run_cost or fixed cost per extra autoclave if we want dollars",
        "best_location": "End of notebook 01 or notebook 02",
    },
])

display(mip_objectives)

### How The MIP Objective Is Written Differently

The constraints may stay mostly the same.

The part that changes is usually `problem.setObjective(...)`.

In [ ]:
mip_code_patterns = pd.DataFrame([
    {
        "objective": "Throughput",
        "code_pattern": "problem.setObjective(sum(x.values()), sense=MAXIMIZE)",
    },
    {
        "objective": "Priority-weighted throughput",
        "code_pattern": "problem.setObjective(sum((1000 + priority[p]) * x[p, a, r, t] for ...), sense=MAXIMIZE)",
    },
    {
        "objective": "Weighted lateness",
        "code_pattern": "problem.setObjective(sum(weight[p] * lateness[p] for p in parts), sense=MINIMIZE)",
    },
    {
        "objective": "Scrap or waste cost",
        "code_pattern": "problem.setObjective(sum(scrap_cost[p] * missed[p] for p in parts), sense=MINIMIZE)",
    },
    {
        "objective": "Extra autoclave penalty",
        "code_pattern": "problem.setObjective(throughput_score - extra_ac_penalty * sum(extra_used[a]), sense=MAXIMIZE)",
    },
])

display(mip_code_patterns)

### MIP Teaching Note

The cleanest student-facing progression is probably:

1. Start with throughput.
2. Add priority-weighted throughput.
3. Discuss lateness as the soft-deadline extension.
4. Ask UME what cost or waste data would be realistic.

Cost and waste are compelling, but they need real assumptions. Otherwise the workshop may look more precise than it actually is.

<hr>

## Drill Routing / VRP Objective Swaps

For the drill-routing model, the core decisions are route decisions:

> Which robot visits which operations, and in what order?

In cuOpt routing, many objective changes are expressed by changing the **cost matrix**, **transit-time matrix**, vehicle costs, or route-level penalties.

In [ ]:
vrp_objectives = pd.DataFrame([
    {
        "objective": "Minimize travel distance",
        "plain_english": "Find the shortest robot path through the drill points.",
        "formulation_shape": "cost[i, j] = distance from i to j",
        "new_data_needed": "coordinates or measured travel distances",
        "best_location": "Notebook 03/03b baseline",
    },
    {
        "objective": "Minimize total cycle time",
        "plain_english": "Minimize travel time plus drilling/service time and any waiting.",
        "formulation_shape": "cost[i, j] = travel_time[i, j] + service_or_setup_time[j]",
        "new_data_needed": "service time, setup time, or process time per operation",
        "best_location": "Notebook 03/03b or 05",
    },
    {
        "objective": "Balance robot workload",
        "plain_english": "Avoid one robot doing all the work while another robot sits idle.",
        "formulation_shape": "MIN max(route_duration[v]) or add route-span penalty",
        "new_data_needed": "multiple robots and comparable route-duration metric",
        "best_location": "Notebook 05 multiple-robot comparison",
    },
    {
        "objective": "Minimize late operations",
        "plain_english": "Prefer routes that finish operations inside target time windows.",
        "formulation_shape": "penalize lateness or constrain arrival windows",
        "new_data_needed": "earliest/latest operation windows or target completion times",
        "best_location": "Notebook 04/05 VRPTW section",
    },
    {
        "objective": "Minimize tool changes or clamp repositioning",
        "plain_english": "A slightly longer path may be better if it avoids expensive setup changes.",
        "formulation_shape": "cost[i, j] = distance[i, j] + setup_penalty[i, j]",
        "new_data_needed": "tool family, clamp zone, setup/changeover penalty",
        "best_location": "Notebook 03/03b as an n+1 drill constraint/objective",
    },
])

display(vrp_objectives)

### How The VRP Objective Is Written Differently

For routing, we often change the numbers cuOpt optimizes over rather than writing an algebraic objective directly.

The most common workshop pattern is:

> build a different matrix, solve the same routing structure, compare the route.

In [ ]:
vrp_code_patterns = pd.DataFrame([
    {
        "objective": "Travel distance",
        "code_pattern": "cost_matrix = distance_matrix_mm",
    },
    {
        "objective": "Cycle time",
        "code_pattern": "cost_matrix = travel_time_ms + service_time_ms_by_destination",
    },
    {
        "objective": "Tool/clamp setup penalty",
        "code_pattern": "cost_matrix = distance_matrix_mm + setup_penalty_matrix",
    },
    {
        "objective": "Time-window compliance",
        "code_pattern": "add transit time + set order time windows; inspect late or infeasible operations",
    },
    {
        "objective": "Balanced workload",
        "code_pattern": "use multiple vehicles and compare route durations; add span/route-duration penalties if supported by chosen API path",
    },
])

display(vrp_code_patterns)

### VRP Teaching Note

The best routing objective swap for this workshop is probably:

> shortest distance vs. lowest cycle time vs. fewer setup changes.

That feels factory-specific without needing a huge dataset.

Balanced workload is useful once the story has multiple robots.

<hr>

## UME Decision Table

This table is the practical decision point.

UME can mark each row as:

- `include now`
- `include as discussion`
- `save for advanced`
- `drop`

In [ ]:
ume_decision_table = pd.DataFrame([
    {
        "candidate": "MIP: priority-weighted throughput",
        "why_it_teaches_well": "Small code change; clear business meaning",
        "risk": "Low",
        "recommended_role": "include now or keep in notebook 01",
        "UME_decision": "TODO",
    },
    {
        "candidate": "MIP: minimize lateness",
        "why_it_teaches_well": "Shows hard vs soft deadline tradeoff",
        "risk": "Medium because out-time lateness may imply scrap, not normal lateness",
        "recommended_role": "discussion plus notebook 02 demo",
        "UME_decision": "TODO",
    },
    {
        "candidate": "MIP: scrap/waste cost",
        "why_it_teaches_well": "Connects optimization to dollars/waste",
        "risk": "Needs real cost assumptions",
        "recommended_role": "discussion unless data is available",
        "UME_decision": "TODO",
    },
    {
        "candidate": "VRP: distance vs cycle time",
        "why_it_teaches_well": "Shows why the cheapest path may not be the fastest process",
        "risk": "Low if service times are simple",
        "recommended_role": "include now if Day 2 has time",
        "UME_decision": "TODO",
    },
    {
        "candidate": "VRP: tool/clamp setup penalty",
        "why_it_teaches_well": "Very factory-specific and intuitive",
        "risk": "Needs credible setup-penalty assumptions",
        "recommended_role": "strong candidate for n+1",
        "UME_decision": "TODO",
    },
    {
        "candidate": "VRP: balanced robot workload",
        "why_it_teaches_well": "Good bridge from TSP to multi-robot VRP",
        "risk": "Medium; objective support depends on exact API pattern we use",
        "recommended_role": "notebook 05 or advanced comparison",
        "UME_decision": "TODO",
    },
])

display(ume_decision_table)

## Where These Might Move Later

Recommended placement after UME reviews:

| Objective Swap | Likely Home |
|---|---|
| Priority-weighted throughput | `01-Autoclave-MIP-Workshop.ipynb` |
| Lateness / soft-deadline model | `02-Autoclave-Formulation-Extensions.ipynb` |
| Scrap or waste cost | PPT discussion first, notebook later |
| Distance vs. cycle time | `03b-Drill-Path-Routing-Workshop.ipynb` or `05-VRP-Runnable-Comparisons.ipynb` |
| Tool/clamp setup penalty | `03b` as a factory-specific n+1 section |
| Balanced robot workload | `05-VRP-Runnable-Comparisons.ipynb` |

Until then, keep this as a review notebook.

## Takeaways

1. Objective swaps are a simple way to teach that optimization is not just finding any feasible plan.
2. MIP objectives are usually written directly with algebraic variables.
3. VRP objectives are often expressed through cost matrices, time matrices, and route-level penalties.
4. The most workshop-ready swaps are priority-weighted throughput for MIP and distance vs. cycle time for VRP.
5. Cost, waste, and setup penalties become stronger once UME confirms realistic data or assumptions.